# Inference Methods

StanBKT can fit the same model with four inference methods. This example fits one dataset with each and compares their speed, estimates and predictions. It covers:

1. Choosing a fit method with `FitMethod` and its options class.
2. How summaries differ between methods.
3. Which prediction paths each method supports.

See the [Inference Methods](../guide/inference.rst) guide page for details.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

# VB and MLE fits also log cmdstanpy deprecation notices, hide them in this example
get_logger().setLevel(logging.ERROR)

In [2]:
from stanbkt.utils import sim_simple_BKT

true_params = {"prior": 0.3, "learn": 0.1, "forget": 0.02, "guess": 0.2, "slip": 0.1}
data_df = sim_simple_BKT(n_students=100, n_problems=30, n_kcs=1, rng_seed=11, **true_params)

## Fit with each method

Each method has its own options class. Passing options of the wrong class raises a `TypeError`.

In [3]:
import time

from stanbkt.fits import FitMethod, MCMCFitOptions, MLEFitOptions, PFFitOptions, VBFitOptions
from stanbkt.models import StandardBKT
from stanbkt.utils import VerbosityLevel

options = {
    FitMethod.MCMC: MCMCFitOptions(seed=1, iter_warmup=500, iter_sampling=500, show_progress=False),
    FitMethod.VB: VBFitOptions(seed=1),
    FitMethod.MLE: MLEFitOptions(seed=1),
    FitMethod.PATHFINDER: PFFitOptions(seed=1),
}

models, timings = {}, {}
for method, opts in options.items():
    model = StandardBKT(fit_method=method, verbose=VerbosityLevel.WARN)
    start = time.perf_counter()
    model.fit(data_df, stan_fit_options=opts)
    timings[method.value] = time.perf_counter() - start
    models[method.value] = model

pd.Series(timings, name="fit seconds").round(2)

mcmc          1.26
vb            0.19
mle           0.01
pathfinder    0.22
Name: fit seconds, dtype: float64

The first fit also compiles the Stan model if it is not cached yet, so time a second run for a fair comparison. On this small dataset all methods are fast; the gap grows with the data.

## Summaries differ by method

MCMC returns CmdStan's full summary with diagnostics, VB and Pathfinder return the mean, standard deviation and percentiles of their approximate draws, and MLE returns a single `value`.

In [4]:
for name, model in models.items():
    print(name, list(model.summary().columns))

mcmc ['Mean', 'MCSE', 'StdDev', 'MAD', '2.5%', '50%', '97.5%', 'ESS_bulk', 'ESS_tail', 'ESS_bulk/s', 'R_hat']
vb ['mean', 'sd', '2.5%', '50%', '97.5%']
mle ['value']
pathfinder ['mean', 'sd', '2.5%', '50%', '97.5%']


In [5]:
point_column = {"mcmc": "Mean", "vb": "mean", "mle": "value", "pathfinder": "mean"}
params = ["pi_know", "learn", "forget", "guess", "slip"]

estimates = pd.DataFrame({
    name: [model.summary().loc["kc_0"].loc[p][point_column[name]].iloc[0] for p in params]
    for name, model in models.items()
}, index=params)
estimates["truth"] = [true_params[k] for k in ["prior", "learn", "forget", "guess", "slip"]]
estimates.round(3)

,mcmc,vb,mle,pathfinder,truth
pi_know,0.430,0.426,0.430,0.430,0.30
learn,0.082,0.087,0.081,0.081,0.10
forget,0.016,0.014,0.015,0.015,0.02
guess,0.196,0.202,0.195,0.197,0.20
slip,0.107,0.103,0.107,0.108,0.10


## Prediction support

Point estimate prediction (`predict`) works for every method. Posterior prediction works for every method with `backend="numba"`. The Stan backend (`generate_quantities`) does not accept Pathfinder fits.

In [6]:
from stanbkt.utils import auc, rmse

rows = []
for name, model in models.items():
    preds = model.predict(data_df)
    post = model.predict_posterior_summary(data_df, backend="numba")
    rows.append({
        "method": name,
        "auc (point)": auc(preds["correct"], preds["pCorrectness"]),
        "rmse (point)": rmse(preds["correct"], preds["pCorrectness"]),
        "rmse (posterior mean)": rmse(post["correct"], post["pCorrectness_mean"]),
    })
pd.DataFrame(rows).set_index("method").round(4)

,auc (point),rmse (point),rmse (posterior mean)
method,,,
mcmc,0.7949,0.3925,0.3925
vb,0.7950,0.3926,0.3926
mle,0.7950,0.3925,0.3925
pathfinder,0.7950,0.3925,0.3925


In [7]:
try:
    models["pathfinder"].predict_posterior_summary(data_df, backend="stan")
except ValueError as e:
    print("ValueError:", e)

ValueError: The Stan backend runs CmdStanPy's generate_quantities, which does not support Pathfinder fits. Use backend='numba', which predicts from the same Pathfinder draws.


For MLE, posterior prediction has a single draw, so its intervals collapse to the point estimate.

## Which to use

- **MCMC** for final results. It is the only method with convergence diagnostics (`R_hat`, ESS).
- **VB** or **Pathfinder** for fast approximate posteriors on large data, or to explore models before an MCMC run.
- **MLE** for quick point estimates. With priors it is a penalized maximum likelihood estimate.